In [0]:
"""

# =========================================================
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# =========================================================


DATASETS:
1. transactions.csv
2. customers.json
3. branches.parquet


1. Read and validate all datasets.
2. Convert transaction_date to DateType.
3. Filter failed transactions.
4. Identify duplicate transaction IDs.
5. Calculate total transaction amount per customer.
6. Find suspicious transactions greater than 1 lakh.
7. Join customers with transactions.
8. Find branches with highest transaction volume.
9. Calculate daily transaction totals.
10. Customers count based on the account type.
11. Use window functions to rank top customers.
12. Save final fraud analysis report.


# =========================================================
# PROJECT 3 - Application and Servers
# =========================================================


DATASETS:
1. application_logs.json
2. server_details.csv


1. Read log files.
2. Extract log_date and log_hour.
3. Filter ERROR logs.
4. Count errors by server.
5. Find most frequent error message.
6. Join logs with server details.
7. Calculate hourly error trend.
8. Save error summary report.

# =========================================================
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# =========================================================

DATASETS:
1. patients.csv
2. appointments.json
3. doctors.parquet

1. Read and validate all datasets.
2. Convert appointment_date to DateType.
3. Filter cancelled appointments.
4. Count appointments by status.
5. Find total consultation fees per patient.
6. Find doctors with highest number of appointments.
7. Join patients with appointments.
8. Join appointments with doctors.
9. Find patients with no appointments (ANTI JOIN).
10. Calculate daily appointment counts.
11. Rank doctors by total consultation revenue.
12. Find the most common specialization.
13. Save final healthcare report.

"""

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 1. Read and validate all datasets.

df_transactions = spark.table("dataframeassigment.etl2.transactions")

df_customers = spark.table("dataframeassigment.etl2.customers")

df_branches = spark.table("dataframeassigment.etl2.branches")

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 1. Read and validate all datasets.

df_transactions.printSchema()
df_customers.printSchema()
df_branches.printSchema()

print(df_transactions.count())
print(df_customers.count())
print(df_branches.count())


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 2. Convert transaction_date to DateType.

from pyspark.sql.functions import to_date, col
df_transactions.withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd")).printSchema()

In [0]:
df_transactions.display()

In [0]:
df_customers.display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 3. Filter failed transactions.

from pyspark.sql.functions import col

df_transactions.filter(col("status")=="failed").display()


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 4. Identify duplicate transaction IDs.
from pyspark.sql.functions import col
df_transactions.groupBy("transaction_id").count().filter(col("count") > 1).orderBy(col("count").desc()).display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 5. Calculate total transaction amount per customer.
df_transactions.groupBy("customer_id").sum("amount").withColumnRenamed("sum(amount)", "total_amount").orderBy("total_amount").display()


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 6. Find suspicious transactions greater than 1 lakh.
df_suspicious = df_transactions.filter(col("amount") > 100000)
df_suspicious.display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 7. Join customers with transactions.
df_customer_transaction = df_customers.join(df_transactions, "customer_id", "inner")
df_customer_transaction.display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 8. Find branches with highest transaction volume.
df_highest_transaction = df_branches.join(df_transactions, "branch_id", "inner")
# df_highest_transaction.display()
df_highest_transaction.groupBy("branch_id").sum("amount").withColumnRenamed("sum(amount)", "total_amount").orderBy(col("total_amount").desc()).display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 9. Calculate daily transaction totals.
df_transactions.groupBy("transaction_date").sum("amount").alias("daily_transaction_totals").orderBy("transaction_date").display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 10. Customers counts based on account type
df_customers.groupBy("account_type").count().display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 11. Use window functions to rank top customers.

from pyspark.sql.functions import rank, col
from pyspark.sql.window import Window

df_customer_spending = df_customer_transaction.groupBy("customer_id").sum("amount").withColumnRenamed("sum(amount)", "total_amount")

window_spec = Window.orderBy(col("total_amount").desc())

df_ranked = df_customer_spending.withColumn("rank", rank().over(window_spec))
                                                                
df_ranked.display()



In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 12. Save final fraud analysis report.
df_fraud_report = df_transactions.groupBy("transaction_type").sum("amount").withColumnRenamed("sum(amount)", "total_amount").filter(col("total_amount") > 80000)
df_fraud_report.display()

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 12. Save final fraud analysis report.
df_fraud_report.write.mode("overwrite").saveAsTable("dataframeassigment.etl2.fraud_report")

In [0]:
# PROJECT 3 - Application and Servers
# 1. Read log files.
df_application_logs = spark.read.table("dataframeassigment.etl2.application_logs")
df_server_logs = spark.read.table("dataframeassigment.etl2.server_logs")

In [0]:
# PROJECT 3 - Application and Servers
# validate the records
df_application_logs.printSchema()
df_server_logs.printSchema()

print(df_application_logs.count())
print(df_server_logs.count())

In [0]:
# PROJECT 3 - Application and Servers
# 2. Extract log_date and log_hour.

from pyspark.sql.functions import to_timestamp, to_date, hour, col

df_application_logs = df_application_logs.withColumn("log_timestamp", to_timestamp(col("log_timestamp"), "M/d/yyyy"))
df_application_logs = df_application_logs.withColumn("log_date", to_date(col("log_timestamp")))
df_application_logs = df_application_logs.withColumn("log_hour", hour(col("log_timestamp")))

df_application_logs.display()



In [0]:
# PROJECT 3 - Application and Servers
# 3. Filter ERROR logs.
df_application_logs.filter(col("log_level")=="error").display()

In [0]:
# PROJECT 3 - Application and Servers
# 4. Count errors by server.

df_Counterror_by_server = df_application_logs.filter(col("log_level")=="error").groupBy("server_id").count()
df_Counterror_by_server.display()

In [0]:
# PROJECT 3 - Application and Servers
# 5. Find most frequent error message.
df_application_logs.filter(col("log_level")=="error").groupBy("message").count().orderBy(col("count").desc()).display()


In [0]:
# PROJECT 3 - Application and Servers
# 6. Join logs with server details.

df_application_logs.join(df_server_logs, "server_id", "inner").display()

In [0]:
# PROJECT 3 - Application and Servers
# 7. Calculate hourly error trend.
df_application_logs.filter(col("log_level")=="error").groupBy("log_hour").count().display()

In [0]:
# PROJECT 3 - Application and Servers
# 8. Save error summary report.
df_error_summary = df_application_logs.groupBy("server_id").count().withColumnRenamed("count","error_count")
df_error_summary.display()

In [0]:
df_error_summary.write.mode("overwrite").saveAsTable("dataframeassigment.etl2.error_summary")

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 1. Read and validate all datasets.

df_patients = spark.read.csv("/Volumes/dataframeassigment/etl3/extrack_transfer_load/patients.csv",header=True, inferSchema=True)


df_appointments = spark.read.json("/Volumes/dataframeassigment/etl3/extrack_transfer_load/appointments.json")

df_doctors = spark.read.csv("/Volumes/dataframeassigment/etl3/extrack_transfer_load/doctors.csv",header=True, inferSchema=True)

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 1. Read and validate all datasets.

df_patients.printSchema()
df_appointments.printSchema()
df_doctors.printSchema()

print(df_patients.count())
print(df_appointments.count())
print(df_doctors.count())

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 2. Convert appointment_date to DateType
from pyspark.sql.functions import to_date, col

df_appointments.withColumn("appointment_date", to_date(col("appointment_date"), "M/d/yyyy")).display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 3. Filter cancelled appointments.
df_appointments.filter(col("status")=="Cancelled").display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 4. Count appointments by status.
df_appointments.groupBy("status").count().display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 5. Find total consultation fees per patient.
df_appointments.groupBy("doctor_id").sum("consultation_fee").alias("total_fee").display()


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 6. Find doctors with highest number of appointments.
df_doctor_appoinmnets = df_appointments.join(df_doctors, "doctor_id", "full")
df_doctor_appoinmnets.groupBy("doctor_id", "doctor_name", "experience_years").count().orderBy(col("count").desc()).display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 7. Join patients with appointments.
df_patients.join(df_appointments, "patient_id", "inner").display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 8. Join appointments with doctors.
df_appointments.join(df_doctors, "doctor_id", "inner").display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 9. Find patients with no appointments (ANTI JOIN).
df_patients.join(df_appointments, "patient_id", "left_anti").display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 10. Calculate daily appointment counts.
df_appointments.groupBy("appointment_date").count().display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 11. Rank doctors by total consultation revenue.

from pyspark.sql.functions import rank, col
from pyspark.sql.window import Window

df_revenue = df_appointments.groupBy("doctor_id").sum("consultation_fee").withColumnRenamed("sum(consultation_fee)", "total_fee")
window_spec = Window.orderBy(col("total_fee").desc())
df_revenue.withColumn("rank", rank().over(window_spec)).display()


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 12. Find the most common specialization.
df_appointments.join(df_doctors, "doctor_id", "inner").groupBy("specialization").count().orderBy(col("count").desc()).display()

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 13. Save final healthcare report.
df_final_report = df_patients.join(df_appointments, "patient_id", "inner").join(df_doctors, "doctor_id", "inner")

In [0]:
df_final_report_clean = df_final_report.drop(df_patients.city)
df_final_report_clean.write.mode("overwrite").parquet("/Volumes/dataframeassigment/etl3/extrack_transfer_load/final_healthcare_report")